# Classifier Bootstrap — synth fonts + real handwriting crops (Phase 4 v2)

Mixes synthetic font glyphs with **GT-labeled real handwriting crops** harvested via CRAFT+GT,
and evaluates on **held-out real writers** — the accuracy number that actually matters.

**Setup:** upload `fonts.zip` and `real_crops.zip` to Drive (share 'Anyone with link'),
paste both links below. Internet ON, GPU ON, Run All.

In [ ]:
# ── 0. download fonts.zip + real_crops.zip from Google Drive ────────────
!pip install -q gdown
import gdown, glob, os, re, zipfile
FONTS_URL = 'https://drive.google.com/file/d/1DnF8UGjGQlZCUPx2F02gIg4nNiqZF5Xh/view?usp=sharing'
REAL_URL  = 'https://drive.google.com/file/d/15LBGltQ8lqQmxs1f1Ba-dFHcQAaR-J3N/view?usp=drive_link'
def gget(url, out):
    fid = next(g for g in re.search(r'/d/([\w-]+)|[?&]id=([\w-]+)', url).groups() if g)
    gdown.download(f'https://drive.google.com/uc?id={fid}', out, quiet=False)
gget(FONTS_URL, '/kaggle/working/fonts.zip'); zipfile.ZipFile('/kaggle/working/fonts.zip').extractall('/kaggle/working/fonts')
gget(REAL_URL,  '/kaggle/working/real.zip');  zipfile.ZipFile('/kaggle/working/real.zip').extractall('/kaggle/working/real')
FONTS = sorted(glob.glob('/kaggle/working/fonts/**/*.ttf', recursive=True))
assert FONTS, 'no fonts'; print('fonts:', len(FONTS))
print('real train dirs:', len(glob.glob('/kaggle/working/real/real_crops/train/*')))


In [ ]:
# ── 1. params ──────────────────────────────────────────────────────────
OUT='/kaggle/working'
ALPHABET=list('אבגדהוזחטיכלמנסעפצקרשת')+list('ךםןףץ'); CLS={c:i for i,c in enumerate(ALPHABET)}
RENDER_PX,OUT_SIZE,PER_CLASS,PAD,SEED=128,64,30,0.18,1234
REAL_OVERSAMPLE_FRAC=0.5   # oversample real so it's ~this fraction of the synth count
EPOCHS,BATCH,LR=30,128,3e-4


In [ ]:
# ── 2. synth rendering + augmentation (same as v1) ──────────────────────
import numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont
rng=np.random.default_rng(SEED)
def render_glyph(font,ch):
    asc,desc=font.getmetrics(); box=font.getbbox(ch)
    if box is None: return None
    w=max(box[2]-box[0],1)+8; h=asc+desc+8
    img=Image.new('L',(w,h),0); ImageDraw.Draw(img).text((4-box[0],4),ch,font=font,fill=255)
    a=np.asarray(img); ys,xs=np.where(a>10)
    return None if len(xs)==0 else a[ys.min():ys.max()+1, xs.min():xs.max()+1]
def augment(ink):
    h,w=ink.shape
    a=rng.uniform(20,45); s=rng.uniform(4,7)
    dx=cv2.GaussianBlur(rng.random((h,w)).astype(np.float32)*2-1,(0,0),s)*a
    dy=cv2.GaussianBlur(rng.random((h,w)).astype(np.float32)*2-1,(0,0),s)*a
    gx,gy=np.meshgrid(np.arange(w),np.arange(h))
    ink=cv2.remap(ink,(gx+dx).astype(np.float32),(gy+dy).astype(np.float32),cv2.INTER_LINEAR,borderValue=0)
    ink=cv2.warpAffine(ink,cv2.getRotationMatrix2D((w/2,h/2),rng.uniform(-8,8),1.0),(w,h),borderValue=0)
    ink=cv2.warpAffine(ink,np.float32([[1,rng.uniform(-1,1)*0.18,0],[0,1,0]]),(w,h),borderValue=0)
    op=rng.choice(['erode','dilate','none'])
    if op!='none':
        k=int(rng.integers(1,4)); ker=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))
        ink=(cv2.erode if op=='erode' else cv2.dilate)(ink,ker)
    if rng.random()<0.3: ink=cv2.GaussianBlur(cv2.dilate(ink,np.ones((2,2),np.float32)),(0,0),0.6)
    return np.clip(cv2.GaussianBlur(ink,(0,0),rng.uniform(0.01,1.2)),0,1)
def compose(ink):
    h,w=ink.shape
    if rng.random()<0.5:
        bg=rng.uniform(0.85,1.0); tex=cv2.GaussianBlur(rng.random((h,w)).astype(np.float32),(0,0),3)
        paper=np.clip(bg-0.06*(tex-tex.mean()),0,1)
    else: paper=np.ones((h,w),np.float32)
    out=paper*(1-ink)+0.05*ink+rng.normal(0,rng.uniform(0,12)/255.0,(h,w))
    return (np.clip(out,0,1)*255).astype(np.uint8)
def square(img,size=OUT_SIZE,pad=PAD,bg=255):
    h,w=img.shape; p=int(max(h,w)*pad); side=max(h,w)+2*p
    c=np.full((side,side),bg,np.uint8); c[(side-h)//2:(side-h)//2+h,(side-w)//2:(side-w)//2+w]=img
    return cv2.resize(c,(size,size),interpolation=cv2.INTER_AREA)


In [ ]:
# ── 3. generate synth set ──────────────────────────────────────────────
from tqdm.auto import tqdm
Xs,ys=[],[]
for fp in tqdm(FONTS,desc='synth'):
    try: font=ImageFont.truetype(fp,RENDER_PX)
    except: continue
    for ch in ALPHABET:
        base=render_glyph(font,ch)
        if base is None: continue
        ink0=base.astype(np.float32)/255.0
        for _ in range(PER_CLASS): Xs.append(square(compose(augment(ink0.copy())))); ys.append(CLS[ch])
Xs=np.stack(Xs); ys=np.asarray(ys,np.int64); print('synth:',len(Xs))


In [ ]:
# ── 4. load real crops (train = unseen-writer split handled at packaging) ─
def load_real(split):
    X,y=[],[]
    for d in sorted(glob.glob(f'/kaggle/working/real/real_crops/{split}/*')):
        idx=int(os.path.basename(d).split('_')[0])
        for f in glob.glob(d+'/*.png'):
            a=np.asarray(Image.open(f).convert('L'))
            if a.shape!=(OUT_SIZE,OUT_SIZE): a=np.asarray(Image.open(f).convert('L').resize((OUT_SIZE,OUT_SIZE)))
            X.append(a); y.append(idx)
    return (np.stack(X),np.asarray(y,np.int64)) if X else (np.empty((0,OUT_SIZE,OUT_SIZE),np.uint8),np.empty(0,np.int64))
Xr,yr=load_real('train'); Xv,yv=load_real('val')
print('real train:',len(Xr),' real val:',len(Xv))
# oversample real so it is ~REAL_OVERSAMPLE_FRAC of synth
if len(Xr):
    factor=max(1,round(len(Xs)*REAL_OVERSAMPLE_FRAC/len(Xr)))
    Xr_os=np.repeat(Xr,factor,axis=0); yr_os=np.repeat(yr,factor)
    print('real oversample factor:',factor,'-> real train used:',len(Xr_os))
else: Xr_os,yr_os=Xr,yr
Xtr=np.concatenate([Xs,Xr_os]); ytr=np.concatenate([ys,yr_os]); print('total train:',len(Xtr))


In [ ]:
# ── 5. model ───────────────────────────────────────────────────────────
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(SEED); device='cuda' if torch.cuda.is_available() else 'cpu'; print(device)
def blk(ci,co): return nn.Sequential(nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co,momentum=0.05),nn.ReLU(True),
                                     nn.Conv2d(co,co,3,padding=1),nn.BatchNorm2d(co,momentum=0.05),nn.ReLU(True),nn.MaxPool2d(2))
nc=len(ALPHABET)
model=nn.Sequential(blk(1,32),blk(32,64),blk(64,128),nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                    nn.Linear(128,256),nn.ReLU(True),nn.Dropout(0.4),nn.Linear(256,nc)).to(device)


In [ ]:
# ── 6. train, validating on REAL held-out writers ──────────────────────
def dl(X,y,sh):
    return DataLoader(TensorDataset(torch.from_numpy(X).float().div_(255).unsqueeze(1),torch.from_numpy(y)),
                      batch_size=BATCH,shuffle=sh,num_workers=2)
tr=dl(Xtr,ytr,True); va=dl(Xv,yv,False) if len(Xv) else None
opt=torch.optim.Adam(model.parameters(),lr=LR); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS)
crit=nn.CrossEntropyLoss(label_smoothing=0.05)
best=0.0; best_cm=None
for ep in range(EPOCHS):
    model.train()
    for xb,yb in tr:
        xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); loss=crit(model(xb),yb); loss.backward(); opt.step()
    sched.step()
    if va is None: print(f'ep{ep+1}'); continue
    model.eval(); correct=0; cm=np.zeros((nc,nc),np.int64)
    with torch.no_grad():
        for xb,yb in va:
            p=model(xb.to(device)).argmax(1).cpu(); correct+=(p==yb).sum().item()
            for t,q in zip(yb.numpy(),p.numpy()): cm[t,q]+=1
    acc=correct/len(Xv); print(f'ep{ep+1:2d}/{EPOCHS}  REAL val_acc={acc:.4f}')
    if acc>=best:
        best,best_cm=acc,cm
        torch.save({'state_dict':model.state_dict(),'classes':ALPHABET,'img':OUT_SIZE},f'{OUT}/classifier.pth')
print('BEST real val_acc =',best)


In [ ]:
# ── 7. confusion on real val + save ────────────────────────────────────
import json, matplotlib.pyplot as plt
json.dump({'real_val_acc':best,'classes':ALPHABET},open(f'{OUT}/metrics.json','w'),ensure_ascii=False,indent=2)
if best_cm is not None:
    cmn=best_cm/best_cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(10,9)); ax.imshow(cmn,cmap='viridis')
    ax.set_xticks(range(nc)); ax.set_xticklabels(ALPHABET); ax.set_yticks(range(nc)); ax.set_yticklabels(ALPHABET)
    ax.set_title('REAL val confusion (row-norm)'); fig.tight_layout(); fig.savefig(f'{OUT}/confusion_matrix.png',dpi=120); plt.show()
print('outputs:',os.listdir(OUT))
